# Bird Detection - YOLOv8

## Setup:
1. Upload `data_detection/` folder to Google Drive > BIRD folder
2. Runtime → Change runtime type → **GPU (T4)**
3. Run all cells in order

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, glob

DRIVE_PATH = '/content/drive/MyDrive/BIRD'
LOCAL_DATA = '/content/data_detection'

if not os.path.exists(LOCAL_DATA):
    print('Copying data from Drive...')
    shutil.copytree(os.path.join(DRIVE_PATH, 'data_detection'), LOCAL_DATA)

fixed = 0
for lbl in glob.glob(os.path.join(LOCAL_DATA, '**/*.txt'), recursive=True):
    with open(lbl, 'r') as f:
        lines = f.readlines()
    new_lines = []
    changed = False
    for line in lines:
        parts = line.split()
        if parts and parts[0] != '0':
            parts[0] = '0'
            changed = True
        new_lines.append(' '.join(parts) + '\n')
    if changed:
        with open(lbl, 'w') as f:
            f.writelines(new_lines)
        fixed += 1

train_count = len(os.listdir(os.path.join(LOCAL_DATA, 'train/images')))
val_count = len(os.listdir(os.path.join(LOCAL_DATA, 'val/images')))
print(f'Train: {train_count}, Val: {val_count}, Labels fixed: {fixed}')

Mounted at /content/drive
Copying data from Drive...
Train: 760, Val: 190, Labels fixed: 0


In [2]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 78.8 MB/s eta 0:00:00


# TRAIN YOLOv8-NANO

In [3]:
import yaml

data_yaml = {
    'path': '/content/data_detection',
    'train': 'train/images',
    'val': 'val/images',
    'names': {0: 'bird'}
}

with open('bird_data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print(open('bird_data.yaml').read())

names:
  0: bird
path: /content/data_detection
train: train/images
val: val/images



In [4]:
from ultralytics import YOLO
import shutil

model = YOLO('yolov8n.pt')

results = model.train(
    data='bird_data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    project='yolo_runs',
    name='bird_yolov8n',
    verbose=True
)

# Save best model to Drive
src = 'yolo_runs/bird_yolov8n/weights/best.pt'
dst = '/content/drive/MyDrive/BIRD/best.pt'
shutil.copy(src, dst)
print(f'Model saved to Drive: {dst}')

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=bird_data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscri

FileNotFoundError: [Errno 2] No such file or directory: 'yolo_runs/bird_yolov8n/weights/best.pt'

# INFERENCE

In [6]:
from ultralytics import YOLO
import matplotlib.pyplot as plt

model = YOLO('yolo_runs/bird_yolov8n/weights/best.pt')

IMAGE_PATH = '/content/data_detection/val/images/24_424.jpg'

results = model.predict(IMAGE_PATH, conf=0.25, imgsz=640)

for r in results:
    boxes = r.boxes
    print(f'Found {len(boxes)} bird(s):')
    for box in boxes:
        cls = r.names[int(box.cls[0])]
        conf = float(box.conf[0])
        xyxy = box.xyxy[0].tolist()
        print(f'  {cls}: {conf:.4f} at {[round(v) for v in xyxy]}')
    annotated = r.plot()
    plt.figure(figsize=(10, 8))
    plt.imshow(annotated)
    plt.axis('off')
    plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'yolo_runs/bird_yolov8n/weights/best.pt'

# BATCH INFERENCE

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo_runs/bird_yolov8n/weights/best.pt')

results = model.predict(
    source='/content/data_detection/val/images',
    conf=0.25,
    imgsz=640,
    save=True,
    project='yolo_runs',
    name='val_predictions'
)

total_birds = sum(len(r.boxes) for r in results)
print(f'Processed {len(results)} images, found {total_birds} bird(s) total')